In [ ]:
!pip install transformers
!pip gdown

ERROR: unknown command "gdown" - maybe you meant "download"


In [ ]:
import gdown

url = 'https://drive.google.com/uc?id=11ZZ5iO9HqahU_T3JP5DtPZi0uJI6eQEj'
output = 'kogpt_comment_reply_model.pt' #pt: pytorch

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=11ZZ5iO9HqahU_T3JP5DtPZi0uJI6eQEj
From (redirected): https://drive.google.com/uc?id=11ZZ5iO9HqahU_T3JP5DtPZi0uJI6eQEj&confirm=t&uuid=ca844103-96f3-479a-ba23-6704d8e6435a
To: /content/kogpt_comment_reply_model.pt
100%|██████████| 501M/501M [00:12<00:00, 40.8MB/s]


'kogpt_comment_reply_model.pt'

In [ ]:
import torch
from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast

# 대화에서 사용할 특별 토큰들을 정의합니다.
Q_TKN = "<usr>" # 사용자 입력을 나타내는 토큰
A_TKN = "<sys>" # 시스템의 응답을 나타내는 토큰
BOS = '</s>'    # 문장의 시작을 나타내는 토큰
EOS = '</s>'    # 문장의 종료를 나타내는 토큰
MASK = '<unused0>'  # 마스크 토큰
SENT = '<unused1>'  # 문장 분류를 위한 토큰
PAD = '<pad>'   # 패딩 토큰

# 사전 훈련된 토크나이저 로드 >> TEXT 모델이 처리할 수 있는 형태로 변환
koGPT2_TOKENIZER = PreTrainedTokenizerFast.from_pretrained('skt/kogpt2-base-v2',
                        bos_token=BOS, eos_token=EOS, unk_token='<unk>',
                        pad_token=PAD, mask_token=MASK)

# 사전 훈련된 모델 구성 (환경설정)
config = GPT2Config.from_pretrained('skt/kogpt2-base-v2')

# 사전 훈련된 모델 로드
model = GPT2LMHeadModel(config)

# 사전 훈련된 모델의 가중치 로드 >> CPU를 사용하여 로드
model.load_state_dict(torch.load('kogpt_comment_reply_model.pt', map_location=torch.device('cpu')))

# 모델 평가 모드 설정
# >> 모델이 학습 중이 아닐 때 동작
model.eval()

# 문장 분류를 위한 토큰 초기화
sent = '0'

# 모델 사용, 답변 생성하는 함수 정의
# >> 주어진 입력 텍스트 기반, 최대 max_length 길이의 응답 생성

def dl_model(input_text, max_length=64):

  response ='' # 응답 초기화

  # 최대 길이만큼 반복 >> 응답 생성
        # 0 Q_TKN + "맛있네요." + SENT + sent + A_TKN + "" > model > response (감)
        # 1 Q_TKN + "맛있네요." + SENT + sent + A_TKN + "감" > model > response (사)
        # 2 Q_TKN + "맛있네요." + SENT + sent + A_TKN + "감사" > model > response (합)
        # 3 Q_TKN + "맛있네요." + SENT + sent + A_TKN + "감사합" > model > response (니)
        # 4 Q_TKN + "맛있네요." + SENT + sent + A_TKN + "감사합니" > model > response (다)
        # 5 Q_TKN + "맛있네요." + SENT + sent + A_TKN + "감사합니다" > model > response (.)
        # 6 Q_TKN + "맛있네요." + SENT + sent + A_TKN + "감사합니다." > model > response (EOS)
        # Q_TKN (질문 토큰), SENT(문장토큰), A_TKN(응답토큰)

  for _ in range(max_length):

        # input_text >> 모델이 처리할 수 있는 형태로 인코딩
        # unsqueeze(dim=0) : 입력을 배치(batch) 차원으로 확장
        input_ids = torch.LongTensor(koGPT2_TOKENIZER.encode(Q_TKN + input_text + SENT + sent + A_TKN + response)).unsqueeze(dim=0)

        # attention_mask : 입력 중 어느 부분이 padding 인지 나타내는 마스크 생성
        # >> model 이 padding 부분을 무시하고 실제 데이터 부분에만 집중하도록
        attention_mask = input_ids != koGPT2_TOKENIZER.pad_token_id

        # 모델 예측
        pred = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = pred.logits # 예측 결과의 로짓(logits) 값

        # 예측 결과 중 가장 확률이 높은 토큰 선택
        gen = koGPT2_TOKENIZER.convert_ids_to_tokens(torch.argmax(pred, dim=-1).squeeze().numpy().tolist())[-1]
        # torch.argmax(pred, dim=-1)를 사용, 각 위치에서 가장 높은 확률 가진 토큰의 index를 선택
        # [-1] : 마지막 토큰을 gen 변수에 저장

        if gen == EOS:
          break

        # 생성된 토큰 >> 응답에 추가.
        # 토큰 시작부분에 있는 공백(_) >> 실제로 공백으로 대체
        response += gen.replace('_', ' ')

  return response.strip() # 응답에서 양쪽 공백 제거

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
import urllib.request
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

def generate_reply(input_text):
   output = dl_model(input_text)
   return output

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/tykimos/tykimos.github.io/master/warehouse/dataset/tarrr_sample_submit.txt",
    filename="tarrr_sample_submit.txt",
)

# 파일을 데이터프레임으로 로드
df = pd.read_csv('tarrr_sample_submit.txt', delimiter='\t')
df

,id,comment,reply
0,1,완전 내 스타일이에요! 가격도 적당하고 위치도 좋고👌,의견 감사합니다.
1,2,맛있긴 한데 양이 너무 적어서 좀... ㅠ,의견 감사합니다.
2,3,완전 내 스타일이에요 ㅠㅠ 여기 매장 분위기도 이쁨,의견 감사합니다.
3,4,한국의 전통 음식을 잘 표현한 것 같아요. 향토음식의 정취가 느껴져 좋았습니다.,의견 감사합니다.
4,5,서빙하는 분이 좀 불친절해서 기분이 좀 그랬어요.,의견 감사합니다.
...,...,...,...
95,96,가성비 최고! 이 가격에 이런 맛은 정말 만족스러워요.,의견 감사합니다.
96,97,와... 여기김치... 말도안됨... ㅁㅊ...,의견 감사합니다.
97,98,주문한 지 40분 넘게 기다려서 음식 나왔네요...,의견 감사합니다.
98,99,"아이들이랑 왔는데, 키즈 메뉴도 생각보다 맛있었어요!",의견 감사합니다.


In [ ]:
for idx, row in df.iterrows():
  comment = row['comment']
  reply = generate_reply(comment)

  print(f'[{idx}]')
  print('comment: ', comment)
  print('reply: ', reply)
  print('--'*100)

[0]
comment:  완전 내 스타일이에요! 가격도 적당하고 위치도 좋고👌
reply:  ▁완전▁내▁스타일이에요!▁고객님의▁의견을▁소중히▁여기며,▁가격과▁위치▁모두▁만족시킬▁수▁있도록▁노력하겠습니다.▁다음에도▁저희▁가게를▁찾아주셔서▁감사합니다.▁다음에도▁저희▁가게를▁찾아주셔서▁더▁나은▁경험을▁드리도록▁하겠습니다.
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
[1]
comment:  맛있긴 한데 양이 너무 적어서 좀... ㅠ
reply:  ▁양에▁대한▁의견▁감사합니다.▁양에▁대한▁의견▁감사합니다.▁더▁많은▁양을▁준비하도록▁하겠습니다.
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
[2]
comment:  완전 내 스타일이에요 ㅠㅠ 여기 매장 분위기도 이쁨
reply:  ▁완전▁내▁스타일이에요!▁매장▁분위기와▁함께▁좋은▁시간▁보내셨다니▁기쁩니다!▁앞으로도▁편안하고▁즐거운▁매장▁환경을▁제공하기▁위해▁노력하겠습니다.
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------